## Gold — `dim_data` (calendário)

**Origem:** gerada proceduralmente (sem fonte externa) → **Destino:** `workspace.gold.dim_data`

- **Modelo:** Star Schema. Dimensão conformada de data, reusada pelas fatos via role-playing (ex.: `fato_obras.sk_data_inicio` e `fato_obras.sk_data_situacao`).
- **Grão:** 1 linha por dia, de **1990-01-01** a **2030-12-31**.
- **Transformações:**
  - Calendário gerado com `sequence` diária entre as datas de corte (nenhuma extração necessária).
  - `sk_data` = inteiro no formato `AAAAMMDD` — surrogate key determinística, estável entre execuções e autoexplicativa.
  - Atributos derivados da própria data: ano, semestre, trimestre, bimestre, mês/dia, dia da semana, semana ISO, `ano_mes`, flag de fim de semana.
  - Nomes de mês e dia da semana em português.
- **Linhagem:** geração determinística em memória → `gold.dim_data`.

In [0]:
%run ../shared/_setup

In [0]:
from datetime import date

from pyspark.sql import functions as F
from data_pipeline import save_table, add_column_comments, add_table_comment
from catalogo.dimensao_data import DIM_DATA_COMMENTS, DIM_DATA_TABLE_COMMENT

In [0]:
TARGET_TABLE = "workspace.gold.dim_data"

# Intervalo do calendário
DATA_INICIO = "1990-01-01"
DATA_FIM = "2030-12-31"

# Nomes em português (índice 1-based alinhado às funções do Spark)
MESES = [
    "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
    "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro",
]
DIAS_SEMANA = [
    "Domingo", "Segunda-feira", "Terça-feira",
    "Quarta-feira", "Quinta-feira", "Sexta-feira", "Sábado",
]

# Contrato de saída gold.dim_data
COLUNAS_ORDENADAS = [
    "sk_data",
    "data",
    "ano",
    "semestre",
    "trimestre",
    "bimestre",
    "mes",
    "nome_mes",
    "dia",
    "dia_semana",
    "nome_dia_semana",
    "semana_do_ano",
    "ano_mes",
    "flag_fim_de_semana",
]

In [0]:
df = spark.sql(
    f"""
    SELECT explode(
        sequence(to_date('{DATA_INICIO}'), to_date('{DATA_FIM}'), interval 1 day)
    ) AS data
    """
)

# Surrogate key determinística: AAAAMMDD
df = df.withColumn("sk_data", F.date_format("data", "yyyyMMdd").cast("int"))

# Atributos hierárquicos derivados da própria data
df = df.withColumn("ano", F.year("data"))
df = df.withColumn("semestre", F.when(F.month("data") <= 6, 1).otherwise(2))
df = df.withColumn("trimestre", F.quarter("data"))
df = df.withColumn("bimestre", (F.floor((F.month("data") - 1) / 2) + 1).cast("int"))
df = df.withColumn("mes", F.month("data"))
df = df.withColumn("nome_mes", F.element_at(F.array(*[F.lit(m) for m in MESES]), F.col("mes")))
df = df.withColumn("dia", F.dayofmonth("data"))
df = df.withColumn("dia_semana", F.dayofweek("data"))
df = df.withColumn(
    "nome_dia_semana",
    F.element_at(F.array(*[F.lit(d) for d in DIAS_SEMANA]), F.col("dia_semana")),
)
df = df.withColumn("semana_do_ano", F.weekofyear("data"))
df = df.withColumn("ano_mes", F.date_format("data", "yyyy-MM"))
df = df.withColumn("flag_fim_de_semana", F.col("dia_semana").isin(1, 7))

df = df.select(*COLUNAS_ORDENADAS)

esperado = (date.fromisoformat(DATA_FIM) - date.fromisoformat(DATA_INICIO)).days + 1
print(f"Gold: {df.count():,} dias gerados | esperado: {esperado:,}")
display(df.limit(10))

In [0]:
save_table(df, TARGET_TABLE)
add_column_comments(
    spark,
    TARGET_TABLE,
    DIM_DATA_COMMENTS
)
add_table_comment(spark, TARGET_TABLE, DIM_DATA_TABLE_COMMENT)
print(f"Tabela {TARGET_TABLE} persistida: {spark.table(TARGET_TABLE).count():,} linhas")

In [0]:
total = spark.table(TARGET_TABLE).count()
distintos_sk = spark.table(TARGET_TABLE).select("sk_data").distinct().count()

# Coerência SK <-> data: sk_data deve reconstruir a própria data
incoerentes = spark.table(TARGET_TABLE).filter(
    F.col("data") != F.to_date(F.col("sk_data").cast("string"), "yyyyMMdd")
).count()

min_max = spark.sql(f"SELECT min(data) AS primeira, max(data) AS ultima FROM {TARGET_TABLE}").collect()[0]
print(f"Total: {total:,} | SK distintos: {distintos_sk:,} | Incoerências SK/data: {incoerentes:,}")
print(f"Intervalo: {min_max['primeira']} a {min_max['ultima']}")
assert total == distintos_sk and incoerentes == 0, "Quebra de unicidade/coerência em dim_data"
assert str(min_max['primeira']) == DATA_INICIO and str(min_max['ultima']) == DATA_FIM, "Intervalo fora do contratado"
display(spark.sql(f"SELECT ano, count(*) AS qtd_dias FROM {TARGET_TABLE} GROUP BY ano ORDER BY ano"))
display(spark.sql(f"SELECT nome_mes, count(*) AS qtd_dias FROM {TARGET_TABLE} GROUP BY nome_mes, mes ORDER BY mes"))
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))